# 方案六：多模型加权投票融合复现

不同建模范式的错误往往不相关，因此对 5 个模型的提交结果做软加权投票：每个模型的权重等于其测试集 F1，统计每个四元组的权重和，达到阈值才保留。

| 候选结果 | 模型 | 权重 |
| --- | --- | --- |
| `data/llm/Result 7b-full-2epocns.csv` | Qwen2.5-7B 全参 SFT | 0.8042 |
| `data/pipeline/Result.csv` | 管道式 Stage1+Stage2 | 0.7651 |
| `data/bert/Result.csv` | BERT+CRF | 0.6198 |
| `data/llm/Result.csv` | Qwen3-14B LoRA | 0.5486 |
| `data/baseline/Result.csv` | 规则基线 | 0.4925 |

**运行前置**：仓库需已随附这 5 个 `Result.csv`，或先运行前面的 Notebook 生成。本方案**不使用 GPU**（纯 pandas 操作）。脚本产物路径由 `src/ensemble.py` 原生硬编码到 `data/ensemble/`。

## Step 0. 环境与依赖

本方案只需要 `pandas`。国内机器**先配清华镜像**：

```bash
pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple
pip install pandas
```

In [ ]:
# 缺失时取消注释
# %pip install -q pandas

import importlib.util
print("pandas:", "就绪" if importlib.util.find_spec("pandas") else "缺失")

In [ ]:
# 统一入口：定位项目根目录（notebook/ 的上一级），把 src/ 加入 import 路径
# 脚本产物路径保持原样（config.py 中的 BASE_DIR/outputs, BASE_DIR/data/...）
import os, sys, subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 脚本产物写入的根目录（与 src/config.py 保持一致）
ARTIFACT_DIR = PROJECT_ROOT

# 默认使用 0 号 GPU；多卡机器可改成 "1" 等
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

print("项目根目录:", PROJECT_ROOT)
print("产物根目录:", ARTIFACT_DIR)

def run(cmd):
    """用当前 Python 解释器执行命令，工作目录为 PROJECT_ROOT。"""
    print(">>", " ".join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], check=True, env=os.environ.copy())

## Step 1. 检查五个模型结果是否齐全

In [ ]:
import pandas as pd

files = {
    "7B 全参 SFT": PROJECT_ROOT / "data/llm/Result 7b-full-2epocns.csv",
    "管道式": PROJECT_ROOT / "data/pipeline/Result.csv",
    "BERT+CRF": PROJECT_ROOT / "data/bert/Result.csv",
    "14B LoRA": PROJECT_ROOT / "data/llm/Result.csv",
    "规则基线": PROJECT_ROOT / "data/baseline/Result.csv",
}

missing = [name for name, p in files.items() if not Path(p).exists()]
for name, p in files.items():
    print(f"{name:12s} exists={Path(p).exists()}  path={p}")

if missing:
    print("\n缺少结果文件:", missing, "请先运行对应 Notebook 或确认仓库已随附 data/ 下产物")
else:
    print("\n五个模型结果齐全，可以融合。")

## Step 2. 查看权重与阈值配置

`src/ensemble.py` 顶部集中定义 `PRED_FILES`（文件 + 权重）与 `WEIGHT_THRESHOLDS`（每个阈值输出一个候选文件）。

In [ ]:
import ensemble as ens

print("模型权重（按分数降序）:")
for path, w in ens.PRED_FILES:
    print(f"  {path.parent.name}/{path.name:42s} weight={w}")
print("当前阈值:", ens.WEIGHT_THRESHOLDS)
print("输出目录:", ens.OUTPUT_DIR)

ens.main()
print("\n融合完成，结果文件:", [str(p.relative_to(PROJECT_ROOT)) for p in ens.OUTPUT_DIR.glob("*.csv")])

## Step 3. 运行加权投票融合

```bash
python src/ensemble.py
```

流程：字段归一化（strip；空值 / `_` / `无` / `nan` 统一为 `_`）→ 按 `(id, A, O, C, P)` 去重 → 统计权重和 → 阈值过滤 → 空 id 回退到最强模型（7B）的完整预测。

In [ ]:
run([sys.executable, "src/ensemble.py"])

## Step 4. 校验各阈值候选文件

`Result_t{阈值}.csv` 均为无表头提交格式，且每个测试 id 至少一行。

In [ ]:
COLS = ["id", "AspectTerms", "OpinionTerms", "Categories", "Polarities"]
CATS = {"整体", "使用体验", "功效", "价格", "物流", "气味", "包装",
        "真伪", "服务", "其他", "成分", "尺寸", "新鲜度"}
POLS = {"正面", "负面", "中性"}
test_ids = set(pd.read_csv("data/raw/TEST/Test_reviews.csv")["id"].astype(int))

rows = []
for p in sorted((ARTIFACT_DIR / "data/ensemble").glob("Result_t*.csv")):
    df = pd.read_csv(p, header=None, names=COLS, dtype=str)
    df["id"] = df["id"].astype(int)
    assert set(df["id"]) == test_ids, f"{p.name} id 覆盖不完整"
    assert set(df["Categories"].str.strip()) <= CATS
    assert set(df["Polarities"].str.strip()) <= POLS
    rows.append({"文件": p.name, "四元组数": len(df),
                 "覆盖id": df["id"].nunique(),
                 "密度(个/条)": round(len(df) / len(test_ids), 2)})
display(pd.DataFrame(rows))

## 阈值实验结论（线上测试分数）

由于 5 个权重两两之和只有 9 个离散取值，阈值落在不同“模型组合带”内结果才不同：

| 阈值 τ | 测试 F1 | 含义 |
| --- | --- | --- |
| 0.80 | 0.7922 | 几乎保留所有预测，7B 独有噪声也进来了（最差） |
| 1.26 | 0.8098 | 剔除弱共识 |
| 1.40 | **0.8116（最优）** | 保留 7B 与强模型的共识 |
| 1.45 | 0.8112 | 与 1.40 接近的候选 |
| 1.60 | 0.8064 | 共识过严，开始误删 |
| 2.00 | 0.8003 | 只留三方以上共识，召回损失大 |

核心发现：**LLM 7B 的独有预测是有害的**；“7B + 管道”“7B + BERT”这类强模型两两共识增益最大；过松会放进单模型噪声，过严会损失弱模型之间仍然正确的三方共识。

## 可选：复现其他阈值的候选文件

当前脚本默认只生成 1.30 / 1.35 / 1.45。若要补齐上表其他阈值，可在本 cell 临时修改模块常量后重跑（结果写到 `notebook/04-ensemble/data/ensemble/sweep/`，不覆盖默认产物）。

In [ ]:
# 可选实验：生成 0.80 / 1.26 / 1.40 / 1.60 / 2.00 五个候选文件
ens.WEIGHT_THRESHOLDS = [0.8, 1.26, 1.4, 1.6, 2.0]
ens.OUTPUT_DIR = ARTIFACT_DIR / "data" / "ensemble" / "sweep"
ens.main()